# Gender Identity Probing — GPT-2 and Pythia-1.4B

**Phase 2:** Gender identity activation probing with exact token matching.

**Hardware:** M4 for GPT-2 (~10 min). A100 for Pythia-1.4B (~20 min).  
**Run all cells top to bottom.**

**Critical:** This notebook uses exact token matching (`token.strip().lower() == target_word.lower()`).  
Substring matching causes `'man'` to match inside `'woman'`, producing ~60% accuracy instead of 100%.  
With exact matching, both models achieve **100.0% ± 0.0%** at all layers.

**Outputs:** `results/gender_probe_gpt2.csv`, `results/gender_probe_pythia14b.csv`


## Cell 1: Setup

In [1]:
!pip install -q numpy==1.26.4
!pip install -q transformer_lens datasets scikit-learn

import torch, os
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from transformer_lens import HookedTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
N_PER_GROUP = 24
RANDOM_SEED = 42
os.makedirs('results', exist_ok=True)
print(f'Device: {DEVICE}')
print('Ready.')


Device: cuda
Ready.


## Cell 2: Load BBQ gender identity dataset

In [2]:
dataset = load_dataset('Elfsong/BBQ', split='gender_identity')
gender_df = dataset.to_pandas()
gender_ambig = gender_df[gender_df['context_condition'] == 'ambig'].reset_index(drop=True)
print(f'Ambiguous prompts available: {len(gender_ambig)}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/age-00000-of-00001.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

data/disability_status-00000-of-00001.pa(…):   0%|          | 0.00/85.2k [00:00<?, ?B/s]

data/gender_identity-00000-of-00001.parq(…):   0%|          | 0.00/217k [00:00<?, ?B/s]

...

Generating race_x_ses split:   0%|          | 0/11160 [00:00<?, ? examples/s]

Generating religion split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating ses split:   0%|          | 0/6864 [00:00<?, ? examples/s]

Generating sexual_orientation split:   0%|          | 0/864 [00:00<?, ? examples/s]

Ambiguous prompts available: 2836


## Cell 3: Helper functions
Note exact matching in `extract_acts_exact` — this is the critical fix.

In [3]:
def format_prompt(row):
    return (
        f"Context: {row['context']}\n"
        f"Question: {row['question']}\n"
        f"A) {row['ans0']}\nB) {row['ans1']}\nC) {row['ans2']}\nAnswer:"
    )

def extract_acts_exact(prompt, model, target_word, layers):
    """
    Extract residual stream activations using EXACT token matching.
    This prevents 'man' from matching inside 'woman' at the same position,
    which causes spurious ~60% probe accuracy from contaminated activations.
    """
    tokens = model.to_tokens(prompt)
    strs = [model.to_string(tokens[0][i]) for i in range(tokens.shape[1])]
    pos = next(
        (i for i, t in enumerate(strs) if t.strip().lower() == target_word.lower()),
        None
    )
    if pos is None:
        return None
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)
    return {
        l: cache[f'blocks.{l}.hook_resid_post'][0, pos, :].detach().cpu().float().numpy()
        for l in layers
    }

def probe_cv(X, y, seed=RANDOM_SEED):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    scores = []
    for train_idx, test_idx in cv.split(X, y):
        clf = LogisticRegression(max_iter=500, solver='lbfgs', random_state=seed)
        clf.fit(X[train_idx], y[train_idx])
        scores.append(clf.score(X[test_idx], y[test_idx]))
    return np.mean(scores), np.std(scores)

print('Helpers defined.')


Helpers defined.


## Cell 4: GPT-2 — extract and probe

In [4]:
GPT2_LAYERS = [3, 6, 9, 11]

print('Loading GPT-2...')
model = HookedTransformer.from_pretrained('gpt2', device=DEVICE, dtype=torch.float32)
model.eval()
print('Loaded.')

group_acts = {'man': [], 'woman': []}
for _, row in gender_ambig.iterrows():
    ctx = row['context']
    for label in ['man', 'woman']:
        if len(group_acts[label]) >= N_PER_GROUP:
            continue
        if f' {label} ' in ctx.lower() or f' {label}.' in ctx.lower():
            acts = extract_acts_exact(format_prompt(row), model, label, GPT2_LAYERS)
            if acts is not None:
                group_acts[label].append(acts)
    if all(len(v) >= N_PER_GROUP for v in group_acts.values()):
        break

n = min(len(group_acts['man']), len(group_acts['woman']))
records = [(a, 0) for a in group_acts['man'][:n]] + [(a, 1) for a in group_acts['woman'][:n]]
print(f'Records: man={n}, woman={n}, total={len(records)}')

print('\nGPT-2 Gender Identity Probe (5-fold CV, chance=50%):')
gpt2_results = []
for layer in GPT2_LAYERS:
    X = np.array([r[0][layer] for r in records])
    y = np.array([r[1] for r in records])
    mean, std = probe_cv(X, y)
    print(f'  Layer {layer}: {mean:.1%} ± {std:.1%}')
    gpt2_results.append({'model': 'GPT-2', 'layer': layer,
                         'accuracy': round(mean, 4), 'std': round(std, 4), 'n': len(records)})

df_gpt2 = pd.DataFrame(gpt2_results)
df_gpt2.to_csv('results/gender_probe_gpt2.csv', index=False)
print('Saved: results/gender_probe_gpt2.csv')
print('Expected: 100.0% ± 0.0% at all layers')

del model
if torch.cuda.is_available(): torch.cuda.empty_cache()


Loading GPT-2...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer
Loaded.
Records: man=24, woman=24, total=48

GPT-2 Gender Identity Probe (5-fold CV, chance=50%):
  Layer 3: 100.0% ± 0.0%
  Layer 6: 100.0% ± 0.0%
  Layer 9: 100.0% ± 0.0%
  Layer 11: 100.0% ± 0.0%
Saved: results/gender_probe_gpt2.csv
Expected: 100.0% ± 0.0% at all layers


## Cell 5: Pythia-1.4B — extract and probe
Requires A100. Skip if running on M4.

In [5]:
PYTHIA_LAYERS = [6, 12, 18, 23]

print('Loading Pythia-1.4B...')
model = HookedTransformer.from_pretrained('EleutherAI/pythia-1.4b', device=DEVICE, dtype=torch.float16)
model.eval()
print('Loaded.')

group_acts = {'man': [], 'woman': []}
for _, row in gender_ambig.iterrows():
    ctx = row['context']
    for label in ['man', 'woman']:
        if len(group_acts[label]) >= N_PER_GROUP:
            continue
        if f' {label} ' in ctx.lower() or f' {label}.' in ctx.lower():
            acts = extract_acts_exact(format_prompt(row), model, label, PYTHIA_LAYERS)
            if acts is not None:
                group_acts[label].append(acts)
    if all(len(v) >= N_PER_GROUP for v in group_acts.values()):
        break

n = min(len(group_acts['man']), len(group_acts['woman']))
records = [(a, 0) for a in group_acts['man'][:n]] + [(a, 1) for a in group_acts['woman'][:n]]
print(f'Records: man={n}, woman={n}, total={len(records)}')

print('\nPythia-1.4B Gender Identity Probe (5-fold CV, chance=50%):')
pythia_results = []
for layer in PYTHIA_LAYERS:
    X = np.array([r[0][layer] for r in records])
    y = np.array([r[1] for r in records])
    mean, std = probe_cv(X, y)
    print(f'  Layer {layer}: {mean:.1%} ± {std:.1%}')
    pythia_results.append({'model': 'Pythia-1.4B', 'layer': layer,
                           'accuracy': round(mean, 4), 'std': round(std, 4), 'n': len(records)})

df_pythia = pd.DataFrame(pythia_results)
df_pythia.to_csv('results/gender_probe_pythia14b.csv', index=False)
print('Saved: results/gender_probe_pythia14b.csv')
print('Expected: 100.0% ± 0.0% at all layers')

del model
if torch.cuda.is_available(): torch.cuda.empty_cache()


Loading Pythia-1.4B...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.93G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model EleutherAI/pythia-1.4b into HookedTransformer
Loaded.
Records: man=24, woman=24, total=48

Pythia-1.4B Gender Identity Probe (5-fold CV, chance=50%):
  Layer 6: 100.0% ± 0.0%
  Layer 12: 100.0% ± 0.0%
  Layer 18: 100.0% ± 0.0%
  Layer 23: 100.0% ± 0.0%
Saved: results/gender_probe_pythia14b.csv
Expected: 100.0% ± 0.0% at all layers


In [7]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset('Elfsong/BBQ', split='race_ethnicity')
race_df = dataset.to_pandas()
race_ambig = race_df[race_df['context_condition'] == 'ambig'].reset_index(drop=True)

group_indices = {'Hispanic': [], 'Black': []}
for i, row in race_ambig.iterrows():
    ctx = row['context']
    if 'Hispanic' in ctx:
        group_indices['Hispanic'].append(i)
    elif 'Black' in ctx and 'African American' not in ctx:
        group_indices['Black'].append(i)

hispanic_rows = race_ambig.iloc[group_indices['Hispanic'][:40]].reset_index(drop=True)
black_rows    = race_ambig.iloc[group_indices['Black'][:40]].reset_index(drop=True)
print(f'Hispanic: {len(hispanic_rows)}, Black: {len(black_rows)}')

Hispanic: 40, Black: 40


In [9]:
import torch
from transformer_lens import HookedTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')
model = HookedTransformer.from_pretrained('gpt2', device=DEVICE, dtype=torch.float32)
model.eval()
print('Loaded.')

Device: cuda
Loaded pretrained model gpt2 into HookedTransformer
Loaded.


In [10]:
def format_prompt(row):
    return (
        f"Context: {row['context']}\n"
        f"Question: {row['question']}\n"
        f"A) {row['ans0']}\nB) {row['ans1']}\nC) {row['ans2']}\nAnswer:"
    )

In [11]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

def get_activation_at_pos(prompt, model, pos, layer):
    """Extract residual stream at a specific absolute or relative position."""
    tokens = model.to_tokens(prompt)
    seq_len = tokens.shape[1]
    # Handle negative indexing
    actual_pos = pos if pos >= 0 else seq_len + pos
    actual_pos = max(0, min(actual_pos, seq_len - 1))
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)
    return cache[f'blocks.{layer}.hook_resid_post'][0, actual_pos, :].float().cpu().numpy()

def get_label_pos(prompt, model, label):
    tokens = model.to_tokens(prompt)
    strs = [model.to_string(tokens[0][i]) for i in range(tokens.shape[1])]
    for i, t in enumerate(strs):
        if label.strip() in t.strip():
            return i
    return None

def probe_at_offset(hispanic_rows, black_rows, model, layer, offset, n=40):
    """
    offset: integer relative to label position, or 'fixed_5', 'fixed_15', 'final'
    """
    X, y = [], []

    rows_h = hispanic_rows.iloc[:n] if hasattr(hispanic_rows, 'iloc') else hispanic_rows[:n]
    rows_b = black_rows.iloc[:n] if hasattr(black_rows, 'iloc') else black_rows[:n]

    for label, rows in [('Hispanic', rows_h), ('Black', rows_b)]:
        for _, row in (rows.iterrows() if hasattr(rows, 'iterrows') else enumerate(rows)):
            if hasattr(row, 'iteritems') or not hasattr(row, '__getitem__'):
                r = row
            else:
                r = row
            prompt = format_prompt(r)

            if isinstance(offset, int):
                label_pos = get_label_pos(prompt, model, label)
                if label_pos is None:
                    continue
                pos = label_pos + offset
            elif offset == 'fixed_5':
                pos = 5
            elif offset == 'fixed_15':
                pos = 15
            elif offset == 'final':
                pos = -1

            act = get_activation_at_pos(prompt, model, pos, layer)
            X.append(act)
            y.append(label)

    if len(set(y)) < 2 or len(X) < 10:
        return None, None

    X = np.array(X)
    y = np.array(y)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr, te in cv.split(X, y):
        scaler = StandardScaler()
        clf = LogisticRegression(max_iter=1000, random_state=42)
        clf.fit(scaler.fit_transform(X[tr]), y[tr])
        scores.append(clf.score(scaler.transform(X[te]), y[te]))
    return np.mean(scores), np.std(scores)

# Run ablation
print("PROBE ABLATION — GPT-2, Hispanic vs Black")
print("="*55)

for layer in [3, 6, 11]:
    print(f"\nLayer {layer}:")

    conditions = [
        (0,          "Exact label position (paper result)"),
        (-1,         "Label - 1 (token before)"),
        (1,          "Label + 1 (token after)"),
        (3,          "Label + 3 (downstream)"),
        ('fixed_5',  "Fixed position 5"),
        ('fixed_15', "Fixed position 15"),
        ('final',    "Final token position"),
    ]

    for offset, name in conditions:
        mean, std = probe_at_offset(hispanic_rows, black_rows, model, layer, offset)
        if mean is not None:
            print(f"  {name}: {mean:.1%} ± {std:.1%}")
        else:
            print(f"  {name}: insufficient data")

PROBE ABLATION — GPT-2, Hispanic vs Black

Layer 3:
  Exact label position (paper result): 100.0% ± 0.0%
  Label - 1 (token before): 72.5% ± 6.4%
  Label + 1 (token after): 100.0% ± 0.0%
  Label + 3 (downstream): 100.0% ± 0.0%
  Fixed position 5: 67.5% ± 4.7%
  Fixed position 15: 100.0% ± 0.0%
  Final token position: 100.0% ± 0.0%

Layer 6:
  Exact label position (paper result): 100.0% ± 0.0%
  Label - 1 (token before): 72.5% ± 6.4%
  Label + 1 (token after): 100.0% ± 0.0%
  Label + 3 (downstream): 100.0% ± 0.0%
  Fixed position 5: 67.5% ± 4.7%
  Fixed position 15: 100.0% ± 0.0%
  Final token position: 100.0% ± 0.0%

Layer 11:
  Exact label position (paper result): 100.0% ± 0.0%
  Label - 1 (token before): 72.5% ± 6.4%
  Label + 1 (token after): 100.0% ± 0.0%
  Label + 3 (downstream): 100.0% ± 0.0%
  Fixed position 5: 67.5% ± 4.7%
  Fixed position 15: 100.0% ± 0.0%
  Final token position: 100.0% ± 0.0%
